In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)

# 집에서 psdt 경로: r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\sample_data.csv'

df = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\train.parquet')
# 해당 df의 메모리사용량 확인용
# df.memory_usage(deep=True).sum() / 1024**3

In [3]:
# 컬럼 분리하기

# 시계열 컬럼군
time_cols = ['timestamp', # 필요해서 추가함
        'wl_t',          # 수위데이터 대표
        'sfc_rain_1h',   # 육상 누적강수데이터 대표
        'imerg_rain_1h', # 레이더 누적강수데이터 대표
        'radar_reflectivity_mean', # 레이더반사도 대표
        'radar_reflectivity_p95', # 레이더반사도 대표
        'target_maxrise_6h' # 종속변수: 수위상승량
        ]

# 수위 컬럼군
wl_cols = ['wl_t',
          'wl_t_minus_1h',
          'wl_t_minus_3h',
          'wl_t_minus_6h',
          'wl_t_minus_12h',
          'wl_t_minus_24h',
          ]

# 강수 컬럼군
rain_cols = [
            # IMERG강우
            'imerg_rain_1h',
            'imerg_rain_6h',
            'imerg_rain_24h',
            # 지상강우
            'sfc_rain_1h',
            'sfc_rain_6h',
            'sfc_rain_24h',
            # 레이더 반사도
            'radar_reflectivity_mean',
            'radar_reflectivity_p95',
            'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

In [ ]:
# 시계열데이터 중에 수위변동값('target_maxrise_6h')에 이상치가 의심됨
cond_min_maxrise = df['target_maxrise_6h'] < -2  # 이거 자체가 불리언 조건문임
print(f"6h이내 수위변동이 -2보다 작은 데이터 개수: {cond_min_maxrise.sum()}")  # 불리언조건문은 sum으로 개수를 셀 수 있음
print()
print("<위 조건에 해당하는 데이터>")
print()
display(df[cond_min_maxrise])
# 수위변동값이 -2 이하인 케이스를 보면 이상치 혼자 다른 관측소 값임
# 이 이상치는 심지어 극한 홍수가 발생한 케이스임에도 수위가 빠졌음
# 해당 관측소값만 모아볼 필요가 있음
print()


# 관측소번호 'ST0007'만 모아보기
cond_ST0007 = df['station_id'] == 'ST0007'
# df.loc[cond_ST0007].sort_values('target_maxrise_6h')
# 수위변동 -8.7은 독보적인 값이 맞음, 주변값들을 볼 필요가 있을듯

# timestamp 기준으로 2103-02-26 00:00 ~ 2103-02-29 00:00 을 모아보겠다.
cond_time = (df['timestamp'] > '2103-02-27 00:00') & (df['timestamp'] < '2103-03-01 00:00')
display(df.loc[cond_ST0007 & cond_time, time_cols].sort_values('timestamp'))
# 이상치 확정
print()
df['target_maxrise_6h'].idxmin()

In [ ]:
# 환경변수 중에 집수면적(cat)과 유역면적(ws)도 하나의 이상치 점이 보임
# 1-1 EDA에서 환경변수는 한 관측소당 하나이므로 관측소 기준으로 groupby 시행

area_gb_df = df.groupby("station_id")[['cat_area', 'wsarea']].first().sort_values("cat_area", ascending = False)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    1,  # 1은 생략가능
    1,  # 1은 생략가능
    figsize = (6, 4)
)

ax.scatter(area_gb_df['cat_area'], area_gb_df['wsarea'])
ax.set_xlabel('catchment area')
ax.set_ylabel('watershed area')

plt.tight_layout()
plt.show()

In [ ]:
# 환경변수 중에서 격자내 평균 하천차수가 혼자 7차인 경우가 있어서 확인필요
df.groupby("station_id")[['cat_area', 'wsarea', 'stream_order']].first().sort_values('stream_order', ascending = False)